# Incident Commander — RFT training on Colab

End-to-end submission notebook for the [OpenEnv hackathon](https://huggingface.co/spaces/vasubhrdwj/incident-commander-openenv). Click **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

**What this notebook does, top to bottom:**
1. Verifies the GPU runtime.
2. Clones the env repo and installs the training extras (Unsloth, TRL, matplotlib).
3. Logs into HF (needed at the end to push the LoRA adapter).
4. Boots the env server as a uvicorn subprocess so training rollouts hit a real `IncidentCommanderEnvironment`.
5. Runs **rejection-sampling fine-tuning (RFT)** — every iteration samples 6 fresh rollouts from the env, keeps the top 2 by reward, and SFTs Llama-3.2-3B-Instruct on those trajectories. Pre/post evaluation runs at the start and end of training.
6. Plots loss + reward curves and a baseline-vs-trained bar chart, saved as PNGs to `assets/`.
7. Pushes the trained adapter to your HF Hub repo (optional).

**Why RFT and not GRPO?** We tried GRPO first (`training/train_grpo.py`); the run regressed (pre=0.736 → post=0.380) because parse failures dominated the group-relative advantage signal. RFT is a more reliable choice on small models because it imitates the top quartile of the policy's own behaviour rather than backpropagating through advantage estimates that can explode. The GRPO script stays in the repo as a documented honest ablation.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone the env repo

Replace `REPO_URL` if you've forked. The simulator + grader + RFT script all live in this repo.

In [ ]:
import os
REPO_URL = os.environ.get('OPENENV_REPO', 'https://github.com/vasubhrdwj/incident-commander-openenv.git')
!rm -rf /content/repo
!git clone {REPO_URL} /content/repo
%cd /content/repo/incident_commander

## 3. Install dependencies

Unsloth's Colab one-liner pulls the right torch / transformers / bitsandbytes for the runtime's CUDA version automatically. We then install our env package in editable mode and matplotlib for the plots.

In [ ]:
!pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip -q install -e /content/repo/incident_commander
!pip -q install matplotlib

## 4. Hugging Face login

Token needs **`write`** scope (you'll push the LoRA adapter at the end). The base model `unsloth/Llama-3.2-3B-Instruct` is public so reading is free. Get a token from https://huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 5. Configure the env server

These env vars are read by `server/app.py` when the server process starts. We pin `IC_TASK_ID=easy_canary_regression` for training. Cross-task generalisation (medium / hard) is checked at eval time.

In [ ]:
%env IC_TASK_ID=easy_canary_regression
%env IC_SEED=0
%env IC_STEP_BUDGET=25
%env HOST=0.0.0.0
%env PORT=8000

## 6. Start the env server in the background

`uvicorn` runs as a subprocess for the notebook lifetime. It picks up the `%env` values from the previous cell.

In [ ]:
import subprocess, time, urllib.request, urllib.error

server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server.app:app',
     '--host', '0.0.0.0', '--port', '8000', '--log-level', 'warning'],
    cwd='/content/repo/incident_commander',
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

for _ in range(60):
    try:
        with urllib.request.urlopen('http://localhost:8000/health', timeout=1) as r:
            if r.status == 200:
                print('env server up at http://localhost:8000')
                break
    except (urllib.error.URLError, TimeoutError):
        time.sleep(1)
else:
    raise RuntimeError('env server failed to start within 60s')

## 7. Run RFT

Default config: 8 outer iterations × 6 rollouts per iter × keep top 2, SFT 2 epochs each. Total wall time on a T4: ~25–40 minutes for Llama-3.2-3B (rollouts dominate; SFT itself is fast).

Pre- and post-training evals each do 3 episodes at temperature 0.7. We deliberately don't eval at T=0.1 — the GRPO regression taught us greedy decoding masks mode collapse.

In [ ]:
!python -m incident_commander.training.train_rft \
  --base-model unsloth/Llama-3.2-3B-Instruct \
  --env-url http://localhost:8000 \
  --iterations 8 \
  --rollouts-per-iter 6 \
  --keep-top-k 2 \
  --sft-epochs 2 \
  --rollout-temperature 0.9 \
  --eval-temperature 0.7 \
  --eval-episodes 3 \
  --score-floor 0.30 \
  --lr 2e-4 \
  --lora-rank 16 \
  --output-dir /content/repo/incident_commander/ic-rft-lora \
  --metrics-json /content/repo/incident_commander/rft_metrics.json

## 8. Generate plots

Reads `rft_metrics.json` and writes four PNGs to `assets/`. These get committed to the repo and embedded in the README per submission criteria.

**Plots produced:**
- `training_loss.png` — SFT loss decreasing over RFT iterations.
- `training_reward.png` — sampled / kept / max episode score over iterations.
- `component_comparison.png` — baseline vs trained against per-component weight ceilings.
- `score_summary.png` — pre vs post-training total score with error bars.

In [ ]:
!python -m incident_commander.training.plot_metrics \
  --metrics /content/repo/incident_commander/rft_metrics.json \
  --out /content/repo/incident_commander/assets \
  --task-id easy_canary_regression

## 9. Inline-display the plots so judges see them in the notebook

In [ ]:
from IPython.display import Image, display
import os
ASSETS = '/content/repo/incident_commander/assets'
for f in ['score_summary.png', 'training_reward.png', 'training_loss.png', 'component_comparison.png']:
    p = os.path.join(ASSETS, f)
    if os.path.exists(p):
        print(f)
        display(Image(p))
    else:
        print(f'(missing: {f})')

## 10. Print the headline numbers

Reads the metrics JSON and prints the pre/post deltas in human-readable form. The judges' rubric explicitly asks for "loss and reward plots from a real run" — this cell makes sure the numbers are in the notebook output too, not just buried in PNGs.

In [ ]:
import json
m = json.load(open('/content/repo/incident_commander/rft_metrics.json'))
pre = m['pre_eval']
post = m['post_eval']
delta = post['mean'] - pre['mean']
print(f"\n{'='*55}")
print(f"  Pre-training:  {pre['mean']:.4f}  ± {pre['std']:.4f}  scores={[round(s, 3) for s in pre['scores']]}")
print(f"  Post-training: {post['mean']:.4f}  ± {post['std']:.4f}  scores={[round(s, 3) for s in post['scores']]}")
print(f"  Delta:         {delta:+.4f}  ({'IMPROVED' if delta > 0 else 'REGRESSED'})")
print(f"  RFT iterations: {len(m['iterations'])}")
print(f"{'='*55}")
if delta <= 0:
    print('\nWARN: training regressed. Likely fixes:')
    print('  - increase rollouts_per_iter (more diverse top-K)')
    print('  - lower lr to 1e-4 (overshooting)')
    print('  - raise score_floor to filter out unlucky rollouts')

## 11. Push the LoRA adapter to HF Hub (optional)

Replace `HF_REPO_ID` with something under your username. This makes the trained adapter reproducible and citeable in the README / pitch.

In [ ]:
from huggingface_hub import HfApi
HF_REPO_ID = 'vasubhrdwj/incident-commander-llama3.2-3b-rft'
api = HfApi()
api.create_repo(HF_REPO_ID, private=False, exist_ok=True)
api.upload_folder(
    folder_path='/content/repo/incident_commander/ic-rft-lora',
    repo_id=HF_REPO_ID,
    commit_message='RFT-trained LoRA on easy_canary_regression',
)
print(f'\nadapter pushed: https://huggingface.co/{HF_REPO_ID}')

## 12. Cleanup

In [ ]:
server_proc.terminate()
print('env server stopped')

## Done

**To finish the submission:**
1. The four PNGs are in `/content/repo/incident_commander/assets/`. Right-click → Download, or commit them back to the repo with `git add assets && git push`.
2. The trained LoRA is on HF Hub at the URL printed above.
3. The `rft_metrics.json` file has all numbers in machine-readable form for the writeup.

**Reproducing this run:** judges can hit `Runtime → Run all` on this notebook with their own HF token; everything is deterministic given a fixed `IC_SEED`.